# AdaptCLIP 0-shot CSV analysis

루트의 `adaptclip_*_0shot.csv` 6개 파일을 불러와서 파일별/class별로 `I_AUROC`, `P_AUROC`, `P_AUPR`를 비교합니다.

In [1]:
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown

try:
    import ipywidgets as widgets
except ImportError:
    widgets = None

ROOT = Path.cwd()
if not (ROOT / "main.py").exists():
    ROOT = ROOT.parent.parent

CSV_FILES = sorted(ROOT.glob("adaptclip_*_0shot.csv"))
OUT_DIR = ROOT / "analysis" / "result" / "adaptclip_0shot_checks"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CSV_FILES

[PosixPath('/Users/taehayeong/Desktop/Git_workspace/lab/adaptclip_btad_0shot.csv'),
 PosixPath('/Users/taehayeong/Desktop/Git_workspace/lab/adaptclip_mad_sim_0shot.csv'),
 PosixPath('/Users/taehayeong/Desktop/Git_workspace/lab/adaptclip_mpdd_0shot.csv'),
 PosixPath('/Users/taehayeong/Desktop/Git_workspace/lab/adaptclip_mvtec_0shot.csv'),
 PosixPath('/Users/taehayeong/Desktop/Git_workspace/lab/adaptclip_mvtec_loco_0shot.csv'),
 PosixPath('/Users/taehayeong/Desktop/Git_workspace/lab/adaptclip_visa_0shot.csv')]

In [2]:
def load_with_checks(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path).copy()
    df["file"] = path.name
    df["auroc_winner"] = "tie"
    df.loc[df["I_AUROC"] > df["P_AUROC"], "auroc_winner"] = "image_auroc"
    df.loc[df["P_AUROC"] > df["I_AUROC"], "auroc_winner"] = "pixel_auroc"
    df["I_minus_P_AUROC"] = df["I_AUROC"] - df["P_AUROC"]
    df["P_minus_I_AUROC"] = df["P_AUROC"] - df["I_AUROC"]
    df["P_AUROC_minus_P_AUPR"] = df["P_AUROC"] - df["P_AUPR"]
    df["abs_P_AUROC_P_AUPR_gap"] = df["P_AUROC_minus_P_AUPR"].abs()
    return df

frames = {path.name: load_with_checks(path) for path in CSV_FILES}
all_df = pd.concat(frames.values(), ignore_index=True)

summary_rows = []
for file_name, df in frames.items():
    avg_i = df["I_AUROC"].mean()
    avg_p = df["P_AUROC"].mean()
    overall = "image_auroc" if avg_i > avg_p else "pixel_auroc" if avg_p > avg_i else "tie"
    top_gap = df.sort_values("abs_P_AUROC_P_AUPR_gap", ascending=False).iloc[0]
    summary_rows.append(
        {
            "file": file_name,
            "class_count": len(df),
            "avg_I_AUROC": avg_i,
            "avg_P_AUROC": avg_p,
            "avg_P_AUPR": df["P_AUPR"].mean(),
            "overall_higher_by_average": overall,
            "pixel_auroc_higher_count": (df["auroc_winner"] == "pixel_auroc").sum(),
            "image_auroc_higher_count": (df["auroc_winner"] == "image_auroc").sum(),
            "tie_count": (df["auroc_winner"] == "tie").sum(),
            "top_gap_class": top_gap["Category"],
            "top_gap_value": top_gap["abs_P_AUROC_P_AUPR_gap"],
        }
    )

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUT_DIR / "summary_by_file.csv", index=False)
summary_df.round(2)

,file,class_count,avg_I_AUROC,avg_P_AUROC,avg_P_AUPR,overall_higher_by_average,pixel_auroc_higher_count,image_auroc_higher_count,tie_count,top_gap_class,top_gap_value
0,adaptclip_btad_0shot.csv,12,95.97,97.65,68.74,pixel_auroc,7,5,0,Woven_068,42.9
1,adaptclip_mad_sim_0shot.csv,10,78.66,97.20,31.44,pixel_auroc,10,0,0,dowel,90.8
2,adaptclip_mpdd_0shot.csv,6,73.55,95.95,25.35,pixel_auroc,5,0,1,bracket_white,95.5
3,adaptclip_mvtec_0shot.csv,15,93.49,90.93,38.31,image_auroc,6,9,0,toothbrush,74.2
4,adaptclip_mvtec_loco_0shot.csv,30,74.19,94.92,28.22,pixel_auroc,29,1,0,usb,88.1
5,adaptclip_visa_0shot.csv,12,84.77,95.68,26.12,pixel_auroc,11,1,0,macaroni2,96.0


## 파일별 상세 보기

In [3]:
def show_file_report(file_name: str, top_n: int = 10):
    df = frames[file_name].copy()
    display(Markdown(f"### {file_name}"))
    display(
        df[[
            "Category", "I_AUROC", "P_AUROC", "P_AUPR", "auroc_winner",
            "I_minus_P_AUROC", "P_minus_I_AUROC", "P_AUROC_minus_P_AUPR",
        ]].round(2)
    )

    pixel_higher = df[df["auroc_winner"] == "pixel_auroc"]
    image_higher = df[df["auroc_winner"] == "image_auroc"]
    ties = df[df["auroc_winner"] == "tie"]

    display(Markdown("#### P_AUROC가 I_AUROC보다 높은 class"))
    display(pixel_higher[["Category", "I_AUROC", "P_AUROC", "P_minus_I_AUROC"]].round(2))

    display(Markdown("#### I_AUROC가 P_AUROC보다 높은 class"))
    display(image_higher[["Category", "I_AUROC", "P_AUROC", "I_minus_P_AUROC"]].round(2))

    if len(ties):
        display(Markdown("#### 동률 class"))
        display(ties[["Category", "I_AUROC", "P_AUROC"]].round(2))

    display(Markdown(f"#### P_AUROC와 P_AUPR 차이 큰 class top {top_n}"))
    display(
        df.sort_values("abs_P_AUROC_P_AUPR_gap", ascending=False)[[
            "Category", "P_AUROC", "P_AUPR", "P_AUROC_minus_P_AUPR", "abs_P_AUROC_P_AUPR_gap",
        ]].head(top_n).round(2)
    )

if widgets is None:
    show_file_report(CSV_FILES[0].name)
else:
    file_picker = widgets.Dropdown(options=list(frames), description="CSV")
    top_n_slider = widgets.IntSlider(value=10, min=3, max=30, step=1, description="Top N")
    widgets.interact(show_file_report, file_name=file_picker, top_n=top_n_slider);

### adaptclip_btad_0shot.csv

,Category,I_AUROC,P_AUROC,P_AUPR,auroc_winner,I_minus_P_AUROC,P_minus_I_AUROC,P_AUROC_minus_P_AUPR
0,Woven_001,100.0,99.8,78.2,image_auroc,0.2,-0.2,21.6
1,Woven_127,94.8,94.5,54.0,image_auroc,0.3,-0.3,40.5
2,Woven_104,98.9,95.2,67.6,image_auroc,3.7,-3.7,27.6
3,Stratified_154,98.1,99.6,77.6,pixel_auroc,-1.5,1.5,22.0
4,Blotchy_099,98.3,99.5,79.7,pixel_auroc,-1.2,1.2,19.8
5,Woven_068,97.0,98.7,55.8,pixel_auroc,-1.7,1.7,42.9
6,Woven_125,100.0,99.5,73.9,image_auroc,0.5,-0.5,25.6
7,Marbled_078,99.1,99.2,75.4,pixel_auroc,-0.1,0.1,23.8
8,Perforated_037,93.7,93.5,62.3,image_auroc,0.2,-0.2,31.2
9,Mesh_114,85.6,93.7,58.0,pixel_auroc,-8.1,8.1,35.7


#### P_AUROC가 I_AUROC보다 높은 class

,Category,I_AUROC,P_AUROC,P_minus_I_AUROC
3,Stratified_154,98.1,99.6,1.5
4,Blotchy_099,98.3,99.5,1.2
5,Woven_068,97.0,98.7,1.7
7,Marbled_078,99.1,99.2,0.1
9,Mesh_114,85.6,93.7,8.1
10,Fibrous_183,98.8,99.1,0.3
11,Matted_069,87.3,99.5,12.2


#### I_AUROC가 P_AUROC보다 높은 class

,Category,I_AUROC,P_AUROC,I_minus_P_AUROC
0,Woven_001,100.0,99.8,0.2
1,Woven_127,94.8,94.5,0.3
2,Woven_104,98.9,95.2,3.7
6,Woven_125,100.0,99.5,0.5
8,Perforated_037,93.7,93.5,0.2


#### P_AUROC와 P_AUPR 차이 큰 class top 10

,Category,P_AUROC,P_AUPR,P_AUROC_minus_P_AUPR,abs_P_AUROC_P_AUPR_gap
5,Woven_068,98.7,55.8,42.9,42.9
1,Woven_127,94.5,54.0,40.5,40.5
9,Mesh_114,93.7,58.0,35.7,35.7
8,Perforated_037,93.5,62.3,31.2,31.2
10,Fibrous_183,99.1,68.8,30.3,30.3
2,Woven_104,95.2,67.6,27.6,27.6
11,Matted_069,99.5,73.6,25.9,25.9
6,Woven_125,99.5,73.9,25.6,25.6
7,Marbled_078,99.2,75.4,23.8,23.8
3,Stratified_154,99.6,77.6,22.0,22.0


## 전체 파일에서 gap이 큰 class

In [4]:
all_df.sort_values("abs_P_AUROC_P_AUPR_gap", ascending=False)[[
    "file", "Category", "I_AUROC", "P_AUROC", "P_AUPR", "auroc_winner",
    "P_AUROC_minus_P_AUPR", "abs_P_AUROC_P_AUPR_gap",
]].head(30).round(2)

,file,Category,I_AUROC,P_AUROC,P_AUPR,auroc_winner,P_AUROC_minus_P_AUPR,abs_P_AUROC_P_AUPR_gap
79,adaptclip_visa_0shot.csv,macaroni2,69.9,98.2,2.2,pixel_auroc,96.0,96.0
24,adaptclip_mpdd_0shot.csv,bracket_white,61.4,99.5,4.0,pixel_auroc,95.5,95.5
19,adaptclip_mad_sim_0shot.csv,dowel,74.0,96.6,5.8,pixel_auroc,90.8,90.8
23,adaptclip_mpdd_0shot.csv,bracket_brown,56.3,92.6,2.9,pixel_auroc,89.7,89.7
67,adaptclip_mvtec_loco_0shot.csv,usb,63.0,93.2,5.1,pixel_auroc,88.1,88.1
51,adaptclip_mvtec_loco_0shot.csv,pcb,61.9,94.4,7.4,pixel_auroc,87.0,87.0
46,adaptclip_mvtec_loco_0shot.csv,end_cap,66.1,95.2,10.0,pixel_auroc,85.2,85.2
43,adaptclip_mvtec_loco_0shot.csv,audiojack,63.5,94.9,11.3,pixel_auroc,83.6,83.6
82,adaptclip_visa_0shot.csv,pcb3,65.3,87.9,4.6,pixel_auroc,83.3,83.3
68,adaptclip_mvtec_loco_0shot.csv,usb_adaptor,72.8,97.1,15.3,pixel_auroc,81.8,81.8


## 분석 결과 CSV 저장

In [5]:
for file_name, df in frames.items():
    stem = Path(file_name).stem
    cols = [
        "Shot", "Category", "I_AUROC", "P_AUROC", "P_AUPR", "auroc_winner",
        "I_minus_P_AUROC", "P_minus_I_AUROC", "P_AUROC_minus_P_AUPR", "abs_P_AUROC_P_AUPR_gap",
    ]
    df[cols].to_csv(OUT_DIR / f"{stem}_class_checks.csv", index=False)
    df.sort_values("abs_P_AUROC_P_AUPR_gap", ascending=False)[cols].head(10).to_csv(
        OUT_DIR / f"{stem}_top_pauroc_paupr_gaps.csv", index=False
    )

summary_df.to_csv(OUT_DIR / "summary_by_file.csv", index=False)
print(f"Saved to: {OUT_DIR}")

Saved to: /Users/taehayeong/Desktop/Git_workspace/lab/analysis/result/adaptclip_0shot_checks
